# Searching a multi-tissue bank

{doc}`cross_modal_retrieval` searched the spatial transcriptomics (ST) of one reference section.
Here the bank holds the ST of every tissue tile from **all public HEST-1k Xenium sections**
that have SIGMMA inputs: 56 sections from 14 organs. Given an H&E tile, SIGMMA looks for ST
tiles whose molecular profile matches it, wherever they come from. If the shared embedding
space is meaningful, the hits should come mostly from the query's own organ, even though the
query's section is excluded from the bank.

In this tutorial we:

1. embed every tile of every bank section in both modalities (cached to disk after the first run),
2. search the whole bank with one H&E query and inspect where the hits come from, and
3. measure, for every organ, how often the top-10 ST hits of its H&E tiles come from the same
   organ, compared with chance.

**Requirements:** a GPU, the UNI2-h weights and a SIGMMA checkpoint (see {doc}`../installation`).
The first run embeds every section and is the slow step; later runs load the cache.

## Paths

Each bank section is a folder `data/hest_1k_<organ>/<section>/` holding the same three files as
in the other tutorials. The organ is read from the folder name.

In [ ]:
from pathlib import Path

DATA_DIR = Path("./data")
QUERY_SECTION = "TENX118"     # lung adenocarcinoma; its H&E is the example query
CKPT = Path("./weights/sigmma.pth")
UNI_DIR = Path("./weights/UNI2-h")
CACHE = Path("sigmma_tutorial_outputs/multi_tissue_bank.npz")

FILES = {"image_file": "hematoxylin_eosin.zarr",
         "indices_file": "index_mask_HAE_stride_256_tile_256.zarr",
         "spatial_file": "spatial_data_processed.h5"}
FOV = 256

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, SequentialSampler
from ml_collections.config_dict import ConfigDict

from sigmma.utils import load_model
from sigmma.datasets import SpatialPatchDataset, multimodal_collate

plt.rcParams["figure.dpi"] = 80
device = torch.device("cuda")

## Bank sections

In [ ]:
rows = []
for d in sorted(DATA_DIR.glob("hest_1k_*/*")):
    if all((d / f).exists() for f in FILES.values()):
        rows.append({"organ": d.parent.name.removeprefix("hest_1k_"), "section": d.name, "dir": d})
sections = pd.DataFrame(rows)
print(f"{len(sections)} sections from {sections.organ.nunique()} organs")
sections.groupby("organ").section.agg(["count", lambda s: ", ".join(s)]).rename(
    columns={"<lambda_0>": "sections"})

## Load the model

Same checkpoint and configuration as in {doc}`multiscale_embeddings`.

In [ ]:
config = ConfigDict({
    "model": {
        "name": "sigmma",
        "checkpoint_path": str(CKPT),
        "pretrained_image_encoder": "UNI",
        "pretrained_image_encoder_ckpt_path": str(UNI_DIR),
        "image_encoder_input_dim": 1536,
        "image_encoder_output_dim": 512,
        "spatial_encoder_input_dim": 384,
        "spatial_encoder_output_dim": 512,
        "num_resolutions": 3,
        "contrastive_temperature": 0.07,
        "gumbel_temperature": 0.5,
        "edge_threshold": 0.5,
        "candidate_top_k": 8,
        "candidate_sim_threshold": 0.5,
        "dropout": 0.1,
        "activation": "leaky_relu",
        "aggregate": "attn_pool",
        "num_graph_layers": 2,
        "image_mlp_hidden_dim": 2048,
        "image_attn_pool_heads": 8,
        "lora": {
            "enabled": True, "r": 16, "alpha": 32, "dropout": 0.05,
            "target_modules": ["qkv", "proj"], "bias": "none", "per_scale": False,
        },
    },
})

model = load_model(
    config,
    pretrained_image_encoder=config.model.pretrained_image_encoder,
    pretrained_ckpt_path=config.model.checkpoint_path,
    train=False,
)
model.to(device).eval();

## Embed the bank

As in {doc}`cross_modal_retrieval`, each 256 × 256 px tile gets an H&E embedding and an ST
(cell graph) embedding; we keep the L2-normalised **macro** scale. Embeddings of all
sections are written to `CACHE`; delete that file to recompute, for example after adding
sections or changing the checkpoint.

In [ ]:
SCALE_IDX = 2   # 0 = micro, 1 = meso, 2 = macro

@torch.no_grad()
def embed_section(d, batch_size=64):
    ds = SpatialPatchDataset(**{k: d / v for k, v in FILES.items()}, num_resolutions=3,
                             type_pretrained_cell_embedding="nemo", latent_embedding_name="nemo_latent")
    dl = DataLoader(ds, batch_size=batch_size, sampler=SequentialSampler(ds),
                    num_workers=8, collate_fn=multimodal_collate)
    img, st, xy = [], [], []
    for batch in dl:
        if batch["image"].numel() == 0:
            continue
        with torch.amp.autocast("cuda"):
            out = model(batch["image"].to(device), batch["spatial_graph"].to(device))
        img.append(torch.nn.functional.normalize(out["image_embedding"][SCALE_IDX].float(), dim=-1).cpu())
        st.append(torch.nn.functional.normalize(out["spatial_embedding"][SCALE_IDX].float(), dim=-1).cpu())
        xy.append(np.stack([batch["metadata"]["x_start"].numpy(), batch["metadata"]["y_start"].numpy()], 1))
    return torch.cat(img).numpy(), torch.cat(st).numpy(), np.concatenate(xy)

if CACHE.exists():
    z = np.load(CACHE, allow_pickle=True)
    img, st, xy, tile_section = z["img"], z["st"], z["xy"], z["section"]
else:
    parts, t0 = [], time.time()
    for i, r in sections.iterrows():
        e = embed_section(r.dir)
        parts.append(e + (np.full(len(e[0]), r.section),))
        print(f"[{i + 1}/{len(sections)}] {r.organ}/{r.section}: {len(e[0]):,} tiles "
              f"({(time.time() - t0) / 60:.1f} min)")
    img, st, xy, tile_section = (np.concatenate(p) for p in zip(*parts))
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    np.savez(CACHE, img=img, st=st, xy=xy, section=tile_section)

bank = pd.DataFrame({"section": tile_section, "x_start": xy[:, 0], "y_start": xy[:, 1]})
bank["organ"] = bank.section.map(sections.set_index("section").organ)
print(f"bank: {len(bank):,} tiles, image {img.shape}, ST {st.shape}")
bank.organ.value_counts().to_frame("tiles").assign(share=lambda d: (d.tiles / d.tiles.sum()).round(3))

The bank is unbalanced: breast contributes about half of all tiles. A search that ignored
the image would therefore return mostly breast tiles. Below, every hit rate is compared with
this chance level.

## One H&E query against the whole bank

We take an H&E tile from `TENX118` (lung) and rank the ST embeddings of all tiles from
**other** sections by cosine similarity. Excluding the query's own section removes its
spatial neighbours, which would otherwise be trivial matches.

In [ ]:
sec_codes, sec_index = np.unique(bank.section.to_numpy(), return_inverse=True)
B_st = torch.from_numpy(st).to(device, torch.float16)
B_img = torch.from_numpy(img).to(device, torch.float16)
B_sec = torch.from_numpy(sec_index).to(device)

@torch.no_grad()
def search(queries, query_sec, bank_emb, k=10, chunk=2048):
    """Top-k bank tiles for each query, never from the query's own section."""
    hits = []
    for i in range(0, len(queries), chunk):
        q = torch.from_numpy(queries[i:i + chunk]).to(device, torch.float16)
        s = q @ bank_emb.T
        own = torch.from_numpy(query_sec[i:i + chunk]).to(device)[:, None] == B_sec[None, :]
        s.masked_fill_(own, float("-inf"))
        hits.append(s.topk(k, dim=1))
    return (torch.cat([h.indices for h in hits]).cpu().numpy(),
            torch.cat([h.values for h in hits]).float().cpu().numpy())

query_rows = np.flatnonzero(bank.section.to_numpy() == QUERY_SECTION)
q = int(np.random.default_rng(0).choice(query_rows))
idx, sim = search(img[[q]], sec_index[[q]], B_st, k=10)
hits = bank.iloc[idx[0]].assign(cos_sim=sim[0].round(3)).reset_index(names="tile")
hits

In [ ]:
def tile_rgb(section, x, y):
    he = zarr.open(str(sections.set_index("section").loc[section, "dir"] / FILES["image_file"]), mode="r")
    return np.transpose(np.asarray(he[:, y:y + FOV, x:x + FOV]), (1, 2, 0))

TOP = 8
fig, axes = plt.subplots(1, TOP + 1, figsize=(2.3 * (TOP + 1), 2.7))
qr = bank.iloc[q]
axes[0].imshow(tile_rgb(qr.section, qr.x_start, qr.y_start))
axes[0].set_title(f"query\n{qr.organ} / {qr.section}", fontsize=9, color="red")
for ax, (_, h) in zip(axes[1:], hits.head(TOP).iterrows()):
    ax.imshow(tile_rgb(h.section, h.x_start, h.y_start))
    ax.set_title(f"{h.organ} / {h.section}\ncos {h.cos_sim:.2f}", fontsize=9)
for ax in axes:
    ax.axis("off")
fig.suptitle("H&E query and the H&E of its top ST hits (the search used the hits' ST, not their H&E)", y=1.05)
plt.tight_layout(); plt.show()

## Organ-level accuracy over all queries

Now every tile in the bank acts as a query once. For each H&E tile we take its top-10 ST
tiles from other sections and record their organs; we do the same in the other direction
(ST query → H&E bank). The chance level for a query is the share of its own organ among the
bank tiles outside its section.

In [ ]:
K = 10
organs = np.array(sorted(bank.organ.unique()))
organ_of = pd.Categorical(bank.organ, categories=organs).codes
n_organ_sec = pd.crosstab(sec_index, organ_of).reindex(columns=range(len(organs)), fill_value=0).to_numpy()
n_organ = n_organ_sec.sum(0)

# chance: share of the query's organ among bank tiles outside the query's section
own_left = n_organ[organ_of] - n_organ_sec[sec_index, organ_of]
chance = own_left / (len(bank) - n_organ_sec[sec_index].sum(1))

summary, mixing = {}, {}
for name, Q, B in [("H&E → ST", img, B_st), ("ST → H&E", st, B_img)]:
    idx, _ = search(Q, sec_index, B, k=K)
    same = (organ_of[idx] == organ_of[:, None]).mean(1)
    per_organ = pd.DataFrame({"organ": bank.organ, "same_organ": same, "chance": chance}) \
        .groupby("organ").mean()
    summary[name] = per_organ
    counts = np.zeros((len(organs), len(organs)))
    np.add.at(counts, (np.repeat(organ_of, K), organ_of[idx].ravel()), 1)
    mixing[name] = pd.DataFrame(counts / counts.sum(1, keepdims=True), index=organs, columns=organs)

table = pd.concat({k: v for k, v in summary.items()}, axis=1).round(3)
table.loc["mean over organs"] = table.mean().round(3)
table

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
w = 0.38
x = np.arange(len(organs))
he2st = summary["H&E → ST"].reindex(organs)
ax.bar(x - w / 2, he2st.same_organ, w, label="SIGMMA, H&E → ST")
ax.bar(x + w / 2, summary["ST → H&E"].reindex(organs).same_organ, w, label="SIGMMA, ST → H&E")
ax.scatter(x, he2st.chance, color="black", marker="_", s=300, zorder=3, label="chance")
ax.set_xticks(x, organs, rotation=45, ha="right")
ax.set_ylabel(f"top-{K} hits from the same organ")
ax.set_ylim(0, 1); ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

The mixing matrix shows where the hits of each organ's H&E queries land. Rows are query
organs, columns are the organs of the top-10 ST hits; each row sums to 1. A strong diagonal
means the H&E of an organ is matched to ST of the same organ; off-diagonal blocks show which
organs SIGMMA finds molecularly similar.

In [ ]:
M = mixing["H&E → ST"]
fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(M.to_numpy(), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(organs)), organs, rotation=45, ha="right")
ax.set_yticks(range(len(organs)), organs)
for i in range(len(organs)):
    for j in range(len(organs)):
        v = M.iat[i, j]
        if v >= 0.05:
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if v > 0.5 else "black")
ax.set_xlabel("organ of top-10 ST hits"); ax.set_ylabel("organ of H&E query")
fig.colorbar(im, ax=ax, shrink=0.8, label="fraction of hits")
plt.tight_layout(); plt.show()

## Notes

- Organs with one or two sections (brain, cervix, heart, prostate, kidney, liver) have few or
  no other same-organ tiles once the query's section is excluded, so their hit rates are
  bounded by what the bank holds. Brain, cervix, heart and prostate have a single section
  each, so their same-organ rate is 0 by construction in this setting.
- "Organ" is the HEST-1k folder grouping; one organ can mix healthy and diseased tissue.
- Check that bank sections were not used to train your checkpoint; otherwise the search is
  partly evaluated on training data.
- Add your own sections by placing them under `data/hest_1k_<organ>/<section>/` and deleting
  `CACHE`.